In [2]:
import os
import pickle
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure, cm

In [3]:
# TODO load in fMRI data and partition by Schaefer Atlas parcels
atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx]
cortex = np.where(atlas_only_brain != 0)[0]

In [4]:
# TODO - load in participant's fMRI data
participant_data_path = "/home/zachkaras/fmri/fmri_model_data/midprocess/133/filtered_func_data_clean.nii.gz"
fmri_data = nib.load(participant_data_path)
scan = fmri_data.get_fdata()



In [5]:
# reshape data to 2d
scan_2d = np.reshape(scan, [scan.shape[0]*scan.shape[1]*scan.shape[2], scan.shape[3]])

In [ ]:
# z-score the data
means = scan_2d.mean(axis=1, keepdims=True)
stds = scan_2d.std(axis=1, keepdims=True)

z_scored_2d_scan = (scan_2d - means) / np.where(stds==0, 1, stds)

In [21]:
# Only looking at voxels in the MNI brain
scan_2d_brain = z_scored_2d_scan[brain_idx, :]

# Only looking at voxels labeled in the Schaefer Atlas
scan_2d_schaefer = scan_2d_brain[cortex,:]

In [ ]:
# TODO - split data into training and prediction so it fits the bootstrapping method
# try running bootstrap